### Packages

In [3]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

### Load Dataset

In [4]:
train_csv = pd.read_csv("train.csv")
test_csv = pd.read_csv("test.csv")
print("train shape: ", train_csv.shape)
print("train head: \n", train_csv.head(5))
print("test shape: ", test_csv.shape)
print("test head: \n", test_csv.head(5))

train shape:  (40000, 3)
train head: 
    id                                             review sentiment
0   1  One of the other reviewers has mentioned that ...  positive
1   2  A wonderful little production. The filming tec...  positive
2   4  Basically there's a family where a little boy ...  negative
3   6  Probably my all-time favorite movie, a story o...  positive
4   7  I sure would like to see a resurrection of a u...  positive
test shape:  (10000, 3)
test head: 
    id                                             review  sentiment
0   3  I thought this was a wonderful way to spend ti...        NaN
1   5  Petter Mattei's "Love in the Time of Money" is...        NaN
2  17  Some films just simply should not be remade. T...        NaN
3  20  An awful film! It must have been up against so...        NaN
4  24  First of all, let's get a few things straight ...        NaN


### Process Data

In [ ]:
train_reviews = train_csv["review"].astype(str).fillna("").tolist()
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    min_df = 3,
    max_df = .9,
    ngram_range=(1,2),
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True,
    max_features=50000,
)
tfidf_train = tfidf.fit_transform(train_reviews)

print("TF-IDF matrix shape:", tfidf_train.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (40000, 244095)
Number of features: 244095


### Create LSA object

In [6]:
def make_lsa(n_comp, data):
    svd = TruncatedSVD(n_components=n_comp, n_iter=15, random_state=42)
    lsa = make_pipeline(svd, Normalizer(copy=False))
    Z_train = lsa.fit_transform(data)
    return lsa, Z_train

### Fit LSA

In [7]:
lsa, Z_train = make_lsa(20, tfidf_train)

### Create KNN object

In [8]:
def make_knn(k, dist):
    return KNeighborsClassifier(n_neighbors=k, metric=dist)


### Fit KNN

In [9]:
# mapping the testing set to the same embedding space so we can use knn
test_reviews = test_csv["review"].astype(str).fillna("").tolist()
tfidf_test = tfidf.transform(test_reviews)
# use same "vocabulary" as training set to create embeddings for testing set
Z_test = lsa.transform(tfidf_test)

y_train = train_csv["sentiment"].values

knn = make_knn(50, "cosine")
# first param = training data "features" 
# second param = target values for training data (class)
knn.fit(Z_train, y_train)

# applying knn to unseen data (i.e. test reviews)
test_preds = knn.predict(Z_test)

### Create submission

In [10]:
submission = pd.DataFrame({"id": test_csv["id"], "sentiment": test_preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Saved submission.csv with shape:", submission.shape)

   id sentiment
0   3  positive
1   5  positive
2  17  negative
3  20  negative
4  24  negative
Saved submission.csv with shape: (10000, 2)


### Questions for consideration

1. How could we potentially improve our model?

2. What are the limitations of KNN in our use case?

3. What can we do to try and find the ideal value for k in KNN?

4. What would happen if we tried to use a distance function other than cosine?